# Intervention Jacobian v0.14 — Controlled Independence of Systemhood Diagnostics
Finite-dimensional causal-control diagnostic. No temporal residual is fitted. We test whether B,C,A,P can be independently manipulated by declared controls.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
I=np.eye(2,dtype=complex); Z=np.array([[1,0],[0,-1]],complex); zero=np.array([1,0],complex); one=np.array([0,1],complex)
def dm(psi): return np.outer(psi,psi.conj())
def entropy(r):
    e=np.linalg.eigvalsh((r+r.conj().T)/2); e=e[e>1e-12]; return -np.sum(e*np.log2(e))
def ptrace2(r,keep):
    q=r.reshape(2,2,2,2); return np.trace(q,axis1=1,axis2=3) if keep==0 else np.trace(q,axis1=0,axis2=2)
def Cmi(r): return (entropy(ptrace2(r,0))+entropy(ptrace2(r,1))-entropy(r))/2
def diagnostics(u,tstar=.55,J0=.1,Estar=1.):
    J,g,p,Esep=u
    th=J*tstar; psi=np.cos(th)*np.kron(zero,zero)-1j*np.sin(th)*np.kron(one,one); r0=dm(psi); U=np.kron(Z,I); r=(1-p)*r0+p*(U@r0@U)
    B=Esep/(Esep+Estar)
    C=Cmi(r)
    A=abs(J)/(abs(J)+abs(g)+J0)
    P=float(np.real(np.vdot(psi,r@psi)))
    return np.array([B,C,A,P])
u0=np.array([.6,.35,.18,.6]); print('baseline [B,C,A,P]=',diagnostics(u0))


## Finite-difference intervention Jacobian

In [ ]:
def jacobian(u):
    M=np.zeros((4,4))
    for j in range(4):
        h=1e-5*max(1,abs(u[j])); up=u.copy(); um=u.copy(); up[j]+=h; um[j]-=h; M[:,j]=(diagnostics(up)-diagnostics(um))/(2*h)
    return M
M=jacobian(u0); rows=['B','C','A','P']; cols=['J','g','p','Esep']; print(np.round(M,5)); print('rank=',np.linalg.matrix_rank(M)); sv=np.linalg.svd(M,compute_uv=False); print('singular values=',sv,' condition=',sv[0]/sv[-1])
plt.figure(figsize=(6,5)); plt.imshow(M,aspect='auto'); plt.xticks(range(4),cols); plt.yticks(range(4),rows); plt.colorbar(label='partial derivative'); plt.tight_layout(); plt.show()


## Dimensionless elasticity Jacobian
Raw derivatives mix control units. Compare fractional responses E_kj=(u_j/I_k) dI_k/du_j where defined.

In [ ]:
I0=diagnostics(u0); E=M*u0[None,:]/I0[:,None]; print(np.round(E,4)); svE=np.linalg.svd(E,compute_uv=False); print('elasticity rank=',np.linalg.matrix_rank(E),' condition=',svE[0]/svE[-1])


## Construct targeted interventions
Solve M delta_u ~= target diagnostic direction using the pseudoinverse. Large control norms or strong off-target residuals indicate poor controllability.

In [ ]:
Mp=np.linalg.pinv(M)
for k,name in enumerate(rows):
    target=np.zeros(4); target[k]=.01; du=Mp@target; achieved=M@du; leak=np.linalg.norm(achieved-target); print(name,'du=',np.round(du,5),' achieved=',np.round(achieved,5),' leakage=',leak,' control norm=',np.linalg.norm(du))


## Remove the artificial independent Esep knob
In the earlier dimer proxy Esep=|J|, so binding is not independently controlled. Recompute the effective 4x3 Jacobian for controls J,g,p with Esep tied to J.

In [ ]:
def diag_tied(v):
    J,g,p=v; return diagnostics(np.array([J,g,p,abs(J)]))
v0=np.array([.6,.35,.18]); Mt=np.zeros((4,3))
for j in range(3):
    h=1e-5; vp=v0.copy(); vm=v0.copy(); vp[j]+=h; vm[j]-=h; Mt[:,j]=(diag_tied(vp)-diag_tied(vm))/(2*h)
print(np.round(Mt,5)); print('rank tied=',np.linalg.matrix_rank(Mt),' (maximum possible 3)'); print('singular values=',np.linalg.svd(Mt,compute_uv=False))


## Locality scan
Rank at one operating point is not enough. Scan several baselines and report conditioning.

In [ ]:
for J in [.25,.6,1.0]:
  for g in [.1,.35,.8]:
    u=np.array([J,g,.18,.6]); Q=jacobian(u); s=np.linalg.svd(Q,compute_uv=False); print('J,g=',J,g,' rank=',np.linalg.matrix_rank(Q),' cond=',s[0]/s[-1])


## Interpretation
With an explicitly independent Esep control the local Jacobian can be full rank, so targeted diagnostic directions are mathematically controllable in this toy construction. But tying Esep back to the dimer coupling J immediately limits control rank to at most three. Thus apparent four-dimensional controllability can be created by adding an artificial knob. The physically important question is whether a real model supplies independent controls/observables, not whether a formal parameterization can be made square.